In [ ]:
# ---------- 1. Price Crosses Above EMA 20 ----------
#df_daily['buy_ema20'] = np.where(df_daily['close'] > df_daily['ema_20'], 1, 0)

# ---------- 2. RSI Oversold Reversal (RSI < 30 and rising) ----------
df_daily['rsi_prev'] = df_daily['rsi_7'].shift(1)
df_daily['buy_rsi'] = np.where((df_daily['rsi_prev'] < 30) & (df_daily['rsi_7'] > df_daily['rsi_prev']), 1, 0)

# ---------- 3. MACD Bullish Crossover ----------
df_daily['macd_prev'] = df_daily['macd'].shift(1)
df_daily['macd_signal_prev'] = df_daily['macd_signal'].shift(1)
df_daily['buy_macd'] = np.where((df_daily['macd_prev'] < df_daily['macd_signal_prev']) & 
                                (df_daily['macd'] > df_daily['macd_signal']), 1, 0)

# ---------- 4. Bullish Engulfing Candlestick ----------
df_daily['buy_engulfing'] = np.where(
    (df_daily['close'] > df_daily['open']) &
    (df_daily['close'].shift(1) < df_daily['open'].shift(1)) &
    (df_daily['close'] > df_daily['open'].shift(1)) &
    (df_daily['open'] < df_daily['close'].shift(1)),
    1, 0
)

# ---------- 5. Breakout from Recent High (e.g., last 20 bars) ----------
df_daily['rolling_high20'] = df_daily['close'].rolling(20).max().shift(1)
df_daily['buy_breakout'] = np.where(df_daily['close'] > df_daily['rolling_high20'], 1, 0)

# ---------- 6. High Volume Up Move ----------
df_daily['vol_ma20'] = df_daily['volume'].rolling(20).mean()
df_daily['buy_volume'] = np.where((df_daily['close'] > df_daily['close'].shift(1)) & 
                                 (df_daily['volume'] > df_daily['vol_ma20']), 1, 0)

# ---------- 7. Bollinger Band Lower Bounce ----------
df_daily['buy_bb'] = np.where((df_daily['close'] < df_daily['bb_lower']) & 
                              (df_daily['close'] > df_daily['close'].shift(1)), 1, 0)

# ---------- 8. Gap Up Confirmation ----------
df_daily['gap_up'] = df_daily['open'] > df_daily['close'].shift(1)
df_daily['buy_gap'] = np.where(df_daily['gap_up'] & (df_daily['close'] > df_daily['open']), 1, 0)

# ---------- 9. Trendline Support Bounce ----------
# Approximate using 20-period SMA as dynamic support
#df_daily['buy_trendline'] = np.where((df_daily['close'].shift(1) < df_daily['sma_20'].shift(1)) & 
#                                    (df_daily['close'] > df_daily['sma_20']), 1, 0)

# ---------- 10. Pivot Point Support Bounce ----------
# Simple pivot calculation: pivot = (high + low + close)/3
df_daily['pivot'] = (df_daily['high'] + df_daily['low'] + df_daily['close']) / 3
df_daily['buy_pivot'] = np.where((df_daily['close'].shift(1) < df_daily['pivot'].shift(1)) &
                                 (df_daily['close'] > df_daily['pivot']), 1, 0)


In [ ]:
# ---------- MOMENTUM INDICATORS ----------
# Relative Strength Index
delta = df_daily['close'].diff()
gain = np.where(delta > 0, delta, 0)
loss = np.where(delta < 0, -delta, 0)
avg_gain = pd.Series(gain).rolling(window=7).mean()
avg_loss = pd.Series(loss).rolling(window=7).mean()
rs = avg_gain / avg_loss
df_daily['rsi_7'] = 100 - (100 / (1 + rs))

# Moving Average Convergence Divergence
ema_7 = df_daily['close'].ewm(span=7, adjust=False).mean()
ema_14 = df_daily['close'].ewm(span=14, adjust=False).mean()
df_daily['macd'] = ema_7 - ema_14
df_daily['macd_signal'] = df_daily['macd'].ewm(span=3, adjust=False).mean()


# ---------- VOLATILITY INDICATORS ----------
# Average True Range
high_low = df_daily['high'] - df_daily['low']
high_close = np.abs(df_daily['high'] - df_daily['close'].shift())
low_close = np.abs(df_daily['low'] - df_daily['close'].shift())
df_daily['tr'] = high_low.combine(high_close, max).combine(low_close, max)
df_daily['atr_14'] = df_daily['tr'].rolling(window=7).mean()

# Bollinger Bands
df_daily['bb_middle'] = df_daily['close'].rolling(20).mean()
df_daily['bb_std'] = df_daily['close'].rolling(10).std()
df_daily['bb_upper'] = df_daily['bb_middle'] + 2 * df_daily['bb_std']
df_daily['bb_lower'] = df_daily['bb_middle'] - 2 * df_daily['bb_std']

# ---------- VOLUME INDICATORS ----------
# Volume moving average
df_daily['vol_sma_20'] = df_daily['volume'].rolling(window=10).mean()